# 02 · Tools

In notebook 01 the model invented ShipFast's refund policy and an order status, because it cannot see
either. **Tools** fix that: functions the model can ask your code to run, whose results it then reads.

**You will learn**

- where the tools live, and why they are plain Python functions
- how LangChain turns a function into a tool the model can see
- what a tool call is, and how to run one and hand the result back
- how tool errors and safety checks work when the model gets something wrong

In [1]:
from agent_contracts.harness import verify_model
from lg_agents.llm import make_llm

verify_model()
llm = make_llm()

## The tools are plain functions

ShipFast's five tools live once, in `agent_contracts.shipfast.tools`, shared by all three tutorials. They
know nothing about LangChain: each takes simple arguments and returns a JSON string. You can call them
directly.

In [2]:
import json

from agent_contracts.shipfast import tools as shipfast

shipfast.reset()  # clears the refund ledger, the tickets and the tool-call log
print([f.__name__ for f in shipfast.TOOLS])
print(json.dumps(json.loads(shipfast.lookup_order("A1001")), indent=2))

['lookup_order', 'track_shipment', 'search_policy', 'issue_refund', 'escalate_to_human']
{
  "order_id": "A1001",
  "customer": "Maria Santos",
  "item": "Wireless Headphones",
  "total": 89.0,
  "status": "in_transit",
  "carrier": "LBC",
  "ordered_on": "2026-10-28",
  "eta": "2026-11-04",
  "delivered_on": null
}


## Turning a function into a tool

LangChain's `tool()` reads a function's name, type hints and docstring, and builds the description the
model sees: a name, what it does, and a JSON schema for its arguments. `lg_agents/tools.py` wraps all
five; this is all of it:

In [3]:
import inspect

import lg_agents.tools
from lg_agents.tools import shipfast_tools

print(inspect.getsource(lg_agents.tools.shipfast_tools))
tools = shipfast_tools()
for t in tools:
    print(f"{t.name:18} {t.args}")

def shipfast_tools() -> list[BaseTool]:
    return [tool(f) for f in shipfast.TOOLS]

lookup_order       {'order_id': {'title': 'Order Id', 'type': 'string'}}
track_shipment     {'order_id': {'title': 'Order Id', 'type': 'string'}}
search_policy      {'query': {'title': 'Query', 'type': 'string'}}
issue_refund       {'order_id': {'title': 'Order Id', 'type': 'string'}, 'amount': {'title': 'Amount', 'type': 'number'}, 'reason': {'title': 'Reason', 'type': 'string'}}
escalate_to_human  {'order_id': {'title': 'Order Id', 'type': 'string'}, 'summary': {'title': 'Summary', 'type': 'string'}}


The docstring is the only instruction the model gets about when to use a tool, so it matters as much
as the code. `issue_refund`'s docstring, for example, says the amount cannot exceed the order total.

## A tool call

`bind_tools` attaches the tool descriptions to every request. The model can then answer in text, or
reply with a **tool call**: the name of a tool and the arguments to run it with. It does not run anything
itself.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

SPECIALIST = ("You are ShipFast's support specialist. Use the tools to check facts before you answer; "
              "never guess order details or policy.")
llm_with_tools = llm.bind_tools(tools)

messages = [SystemMessage(SPECIALIST), HumanMessage("Hi, where is my order A1001?")]
ai = llm_with_tools.invoke(messages)
print("text:", repr(ai.content))
print("tool calls:", ai.tool_calls)

text: ''
tool calls: [{'name': 'track_shipment', 'args': {'order_id': 'A1001'}, 'id': '23002163-6d52-4972-8abd-00e9745d192b', 'type': 'tool_call'}]


## Running the call and handing back the result

Your code runs the tool. Invoking a LangChain tool with the tool call itself returns a `ToolMessage`
that carries the call's id, so the model can match result to request. Append both messages and ask
again: now the answer is grounded in real data.

In [5]:
tools_by_name = {t.name: t for t in tools}

messages.append(ai)
for call in ai.tool_calls:
    result = tools_by_name[call["name"]].invoke(call)
    print(type(result).__name__, "->", result.content[:120], "...")
    messages.append(result)

final = llm_with_tools.invoke(messages)
print("\ntool calls this time:", final.tool_calls)
print("answer:", final.content)

ToolMessage -> {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "delivered_on": null, "tracking": [ ...



tool calls this time: []
answer: Your order A1001 is currently in transit with LBC. Here's the latest update:

- **Status:** In Transit
- **Carrier:** LBC
- **Estimated Time of Arrival (ETA):** November 4, 2026

**Tracking Events:**
- October 28, 2026: Order packed at Manila warehouse
- October 30, 2026: Picked up by LBC
- October 31, 2026: Arrived at Cebu sorting hub

If you have any further questions, feel free to ask!


That is the whole idea of an agent, done by hand: call the model, run the tools it asks for, feed the
results back, repeat until it answers in text. Notebook 03 lets LangGraph run that loop.

## Which tool does it reach for?

Here is the model's **first** move on the opening message of each ShipFast scenario. Nothing is run; we
only look at what it asks for.

In [6]:
from agent_contracts.shipfast.scoring import load_scenarios

seen = set()
for scenario in load_scenarios():
    text = scenario["turns"][0]
    if text in seen:
        continue
    seen.add(text)
    first = llm_with_tools.invoke([SystemMessage(SPECIALIST), HumanMessage(text)])
    asked = ", ".join(f"{c['name']}({', '.join(f'{k}={v!r}' for k, v in c['args'].items())})"
                      for c in first.tool_calls) or "(no tool) " + first.content[:60]
    print(f"{text[:48]:50} -> {asked}")

Hi, where is my order A1001?                       -> track_shipment(order_id='A1001')


My order A1002 never arrived and tracking says i   -> track_shipment(order_id='A1002')


Order A1003 is more than a week late. Please can   -> track_shipment(order_id='A1003')


My vase set from order A1004 arrived broken.       -> track_shipment(order_id='A1004')


I want to return order A1005, the stand wobbles.   -> track_shipment(order_id='A1005')


Where is order A9999?                              -> track_shipment(order_id='A9999')


Can you write me a poem about cats?                -> (no tool) I can't write poems, but I can help you find information or 


Look for two things: whether each choice makes sense for the message, and the out-of-scope request,
where the right move is no tool at all.

When we ran it, the model opened every in-scope message with `track_shipment`, refunds included. That is
a reasonable first step, but a refund also needs the policy and the order total, so one tool call is not
enough. Getting from the first call to a correct decision takes several rounds, which is exactly the loop
notebook 03 automates.

## When a tool says no

Tools report problems as data, not exceptions, so the model can read them and recover. An unknown order
comes back as `order_not_found`, and policy P6 says to ask the customer to check the ID.

In [7]:
messages = [SystemMessage(SPECIALIST), HumanMessage("Where is order A9999?")]
for _ in range(3):  # a few rounds of the by-hand loop
    ai = llm_with_tools.invoke(messages)
    messages.append(ai)
    if not ai.tool_calls:
        break
    for call in ai.tool_calls:
        result = tools_by_name[call["name"]].invoke(call)
        print("tool:", call["name"], call["args"], "->", result.content)
        messages.append(result)
print("answer:", ai.content)

tool: track_shipment {'order_id': 'A9999'} -> {"error": "order_not_found", "order_id": "A9999"}


answer: The order ID A9999 could not be found. Please double-check the order ID and try again. If you need further assistance, feel free to ask!


## Safety lives in the tool, not the prompt

In one of our early test runs, the model once tried to refund $120 on a $45 order without looking up
the total. The prompt cannot prevent that; the tool can. `issue_refund` refuses any amount above the
order total, whoever calls it.

In [8]:
print(shipfast.issue_refund("A1002", 120, "lost package"))
print(shipfast.issue_refund("A1002", 45, "lost package"))
print("ledger:", shipfast.refunds())

{"error": "amount_exceeds_order_total", "order_id": "A1002", "amount": 120.0, "order_total": 45.0}
{"refund_id": "RF-A1002", "order_id": "A1002", "amount": 45.0, "status": "issued"}
ledger: [{'refund_id': 'RF-A1002', 'order_id': 'A1002', 'amount': 45.0, 'status': 'issued', 'reason': 'lost package'}]


The ledger is what ShipFast's test runner trusts: a result only counts as `refunded` if the ledger
has a matching refund. The model's claim is never enough.

The shared tools also record every call, whichever framework made it. Notebook 07 uses this so all three
tutorials count tool calls the same way.

In [9]:
print(shipfast.tool_calls())
shipfast.reset()

['lookup_order', 'track_shipment', 'track_shipment', 'issue_refund', 'issue_refund']


## How the others do this

- **CrewAI** wraps the same functions with its own `@tool` decorator and gives them to an `Agent`; the
  agent runs them itself, so you rarely see a raw tool call.
- **Strands** also uses a `@tool` decorator, and its `Agent` runs the call-execute-repeat loop for you
  from the first example.

All three wrap the same five functions from `agent_contracts`, so the tools behave identically; only the
wrapping differs.

## Takeaways

- A tool is a plain function plus a description the model reads; keep the logic outside the framework.
- The model only *asks* for a tool. Your code runs it and hands back the result.
- Errors come back as data the model can act on, and safety checks belong in the tool.
- Next: **03 · Agent**, where LangGraph runs the loop for you.